# Autoencoder Training Notebook (BatchNorm + Dropout Sweep)

This notebook compares several BatchNorm + dropout variants on the curated x64 benchmark split. By default it reuses the saved sweep artifacts from its local artifact folder, selects the best saved dropout run, and only reruns the sweep when `FORCE_RERUN_DROPOUT_SWEEP = True`.

## Submission Context

- Dataset notebook: `data/dataset/x64/benchmark_50k_5pct/notebook.ipynb`
- Dataset config: `data/dataset/x64/benchmark_50k_5pct/data_config.toml`
- Experiment config: `experiments/anomaly_detection/autoencoder/x64/batchnorm_dropout/train_config.toml`
- Artifact root: `experiments/anomaly_detection/autoencoder/x64/batchnorm_dropout/artifacts/autoencoder_batchnorm_dropout`
- Default behavior: load the saved sweep summary, selected checkpoint, and score-ablation outputs if they already exist; only rerun the sweep when explicitly requested.

### Imports

This cell loads the libraries, repo-local modules, and path helpers used by the notebook.

In [ ]:
from pathlib import Path
import json
import random
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score
import torch
from IPython.display import display
from torch.utils.data import DataLoader
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError('Could not locate repo root containing src/wafer_defect and configs/')
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
from wafer_defect.config import load_toml
from wafer_defect.data.wm811k import WaferMapDataset
from wafer_defect.models.autoencoder import ConvAutoencoder
from wafer_defect.scoring import absolute_error_map, squared_error_map, spatial_mean, topk_spatial_mean
from wafer_defect.training.autoencoder import run_autoencoder_epoch

### Run Controls

This cell defines the experiment config path, the dropout sweep values, and the rerun flags. Leave both flags `False` to reuse saved artifacts when they already exist.

In [ ]:
CONFIG_PATH = REPO_ROOT / 'experiments/anomaly_detection/autoencoder/x64/batchnorm_dropout/train_config.toml'
EPOCHS_OVERRIDE = None
RETRAIN = False
RERUN_SCORE_ABLATION = False
ANOMALY_SCORE_NAME = 'topk_abs_mean'
TOPK_RATIO = 0.01
DROPOUT_SWEEP = [0.0, 0.05, 0.1, 0.2]
config = load_toml(CONFIG_PATH)
if EPOCHS_OVERRIDE is not None:
    config['training']['epochs'] = int(EPOCHS_OVERRIDE)
config

### Reproducibility And Helpers

This cell sets the random seed, resolves the execution device, and defines a helper for saving figures.

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def resolve_device(device_name: str) -> torch.device:
    if device_name == 'auto':
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    return torch.device(device_name)

def save_figure(fig: plt.Figure, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, bbox_inches='tight', dpi=150)
    print(f'Saved figure to {path}')
    return path
set_seed(int(config['run']['seed']))
device = resolve_device(config['training']['device'])
device

### Metadata Check

This cell loads the configured metadata CSV so we can verify the split before building loaders.

In [ ]:
try:
    metadata_path = REPO_ROOT / config['data']['metadata_csv']
    image_size = int(config['data'].get('image_size', 64))
    metadata = pd.read_csv(metadata_path)
    display(metadata.head())
    display(metadata['split'].value_counts().rename_axis('split').to_frame('count'))
    display(metadata['is_anomaly'].value_counts().rename_axis('is_anomaly').to_frame('count'))
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "metadata_path = repo_root / config['data']['metadata_csv']\nimage_size = int(config['data'].get('image_size', 64))\nmetadata = pd.read_csv(metadata_path)\ndisplay(metadata.head())\ndisplay(metadata['split'].value_counts().rename_axis('split').to_frame('count'))\ndisplay(metadata['is_anomaly'].value_counts().rename_axis('is_anomaly').to_frame('count'))"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Data Loaders

This cell builds the train, validation, and test loaders used throughout the notebook.

In [ ]:
try:
    train_dataset = WaferMapDataset(metadata_path, split='train', image_size=image_size)
    val_dataset = WaferMapDataset(metadata_path, split='val', image_size=image_size)
    test_dataset = WaferMapDataset(metadata_path, split='test', image_size=image_size)
    train_loader = DataLoader(train_dataset, batch_size=int(config['data']['batch_size']), shuffle=True, num_workers=int(config['data']['num_workers']))
    val_loader = DataLoader(val_dataset, batch_size=int(config['data']['batch_size']), shuffle=False, num_workers=int(config['data']['num_workers']))
    test_loader = DataLoader(test_dataset, batch_size=int(config['data']['batch_size']), shuffle=False, num_workers=int(config['data']['num_workers']))
    print(f'train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "train_dataset = wafermapdataset(metadata_path, split='train', image_size=image_size)\nval_dataset = wafermapdataset(metadata_path, split='val', image_size=image_size)\ntest_dataset = wafermapdataset(metadata_path, split='test', image_size=image_size)\ntrain_loader = dataloader(train_dataset, batch_size=int(config['data']['batch_size']), shuffle=true, num_workers=int(config['data']['num_workers']))\nval_loader = dataloader(val_dataset, batch_size=int(config['data']['batch_size']), shuffle=false, num_workers=int(config['data']['num_workers']))\ntest_loader = dataloader(test_dataset, batch_size=int(config['data']['batch_size']), shuffle=false, num_workers=int(config['data']['num_workers']))\nprint(f'train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}')"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Model Factory

This cell defines how each dropout variant is named on disk and how the corresponding autoencoder is constructed.

In [ ]:
def dropout_slug(dropout_prob: float) -> str:
    return f'dropout_{dropout_prob:.2f}'.replace('.', 'p')

def build_autoencoder(dropout_prob: float) -> ConvAutoencoder:
    return ConvAutoencoder(latent_dim=int(config['model']['latent_dim']), image_size=image_size, use_batchnorm=bool(config['model'].get('use_batchnorm', False)), dropout_prob=float(dropout_prob)).to(device)
print({'dropout_sweep': DROPOUT_SWEEP, 'artifact_root': str(REPO_ROOT / config['run']['output_dir'])})
build_autoencoder(DROPOUT_SWEEP[0])

### Dropout Sweep Or Artifact Reuse

This cell either reuses the saved sweep artifacts or reruns the dropout sweep when explicitly requested.

In [ ]:
try:
    epochs = int(config['training']['epochs'])
    patience = int(config['training'].get('early_stopping_patience', 0))
    min_delta = float(config['training'].get('early_stopping_min_delta', 0.0))
    checkpoint_every = int(config['training'].get('checkpoint_every', 5))
    base_output_dir = REPO_ROOT / config['run']['output_dir']
    base_output_dir.mkdir(parents=True, exist_ok=True)
    sweep_results_path = base_output_dir / 'dropout_sweep_results.csv'
    sweep_summary_path = base_output_dir / 'results' / 'dropout_sweep_summary.json'
    training_ran = False
    resume_from = ''
    sweep_histories = {}
    history = []
    best_state_dict = None
    best_epoch = 0
    best_val_loss = float('inf')
    stale_epochs = 0

    def resolve_run_output_dir(dropout_prob: float, stored_relative: str | None=None) -> Path:
        expected = base_output_dir / dropout_slug(dropout_prob)
        if expected.exists() or not stored_relative:
            return expected
        candidate = REPO_ROOT / Path(stored_relative)
        if candidate.exists():
            return candidate
        return expected
    artifacts_ready = False
    if not RETRAIN and sweep_results_path.exists() and sweep_summary_path.exists():
        sweep_results_df = pd.read_csv(sweep_results_path)
        sweep_summary = json.loads(sweep_summary_path.read_text(encoding='utf-8'))
        best_dropout = float(sweep_summary.get('selected_dropout', sweep_results_df.sort_values('best_val_loss').iloc[0]['dropout_prob']))
        normalized_rows = []
        for row in sweep_results_df.to_dict(orient='records'):
            dropout_prob = float(row['dropout_prob'])
            run_dir = resolve_run_output_dir(dropout_prob, row.get('output_dir'))
            row['dropout_prob'] = dropout_prob
            row['output_dir'] = run_dir.relative_to(REPO_ROOT).as_posix()
            normalized_rows.append(row)
            history_candidate = run_dir / 'results' / 'history.json'
            if history_candidate.exists():
                sweep_histories[dropout_prob] = json.loads(history_candidate.read_text(encoding='utf-8'))
        sweep_results_df = pd.DataFrame(normalized_rows).sort_values(['best_val_loss', 'epochs_ran']).reset_index(drop=True)
        output_dir = resolve_run_output_dir(best_dropout, sweep_summary.get('selected_output_dir'))
        history_path = output_dir / 'results' / 'history.json'
        best_model_path = output_dir / 'checkpoints' / 'best_model.pt'
        artifacts_ready = history_path.exists() and best_model_path.exists()
        if artifacts_ready:
            history = json.loads(history_path.read_text(encoding='utf-8'))
            config['model']['dropout_prob'] = best_dropout
            config['run']['output_dir'] = output_dir.relative_to(REPO_ROOT).as_posix()
            best_checkpoint = torch.load(best_model_path, map_location=device)
            best_epoch = int(best_checkpoint.get('best_epoch', best_checkpoint.get('epoch', 0)))
            best_val_loss = float(best_checkpoint.get('best_val_loss', float('nan')))
            stale_epochs = int(best_checkpoint.get('stale_epochs', 0))
            model = build_autoencoder(best_dropout)
            model.load_state_dict(best_checkpoint['model_state_dict'])
            optimizer = torch.optim.Adam(model.parameters(), lr=float(config['training']['learning_rate']), weight_decay=float(config['training']['weight_decay']))
            best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(f'Found existing sweep artifacts in {base_output_dir}. Skipping sweep.')
        else:
            print('Found sweep metadata, but the selected run artifacts are incomplete. RETRAIN is False, so training will stay skipped and downstream review cells may be unavailable.')
    if RETRAIN:
        print({'epochs': epochs, 'anomaly_score': ANOMALY_SCORE_NAME, 'topk_ratio': TOPK_RATIO, 'dropout_sweep': DROPOUT_SWEEP})
        sweep_rows = []
        sweep_histories = {}
        best_run = None
        for dropout_prob in DROPOUT_SWEEP:
            run_output_dir = base_output_dir / dropout_slug(dropout_prob)
            run_output_dir.mkdir(parents=True, exist_ok=True)
            run_config = json.loads(json.dumps(config))
            run_config['model']['dropout_prob'] = float(dropout_prob)
            run_config['run']['output_dir'] = run_output_dir.relative_to(REPO_ROOT).as_posix()
            model = build_autoencoder(dropout_prob)
            optimizer = torch.optim.Adam(model.parameters(), lr=float(run_config['training']['learning_rate']), weight_decay=float(run_config['training']['weight_decay']))
            history = []
            run_best_val_loss = float('inf')
            run_best_epoch = 0
            run_best_state_dict = None
            run_stale_epochs = 0
            print(f'\n=== Dropout {dropout_prob:.2f} | output={run_output_dir} ===')
            for epoch in range(epochs):
                train_metrics = run_autoencoder_epoch(model, train_loader, device, optimizer)
                val_metrics = run_autoencoder_epoch(model, val_loader, device)
                record = {'epoch': epoch + 1, 'train_loss': train_metrics.loss, 'val_loss': val_metrics.loss}
                history.append(record)
                print({'dropout': dropout_prob, **record})
                improved = run_best_val_loss - val_metrics.loss > min_delta
                if improved:
                    run_best_val_loss = val_metrics.loss
                    run_best_epoch = epoch + 1
                    run_best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                    run_stale_epochs = 0
                    torch.save({'epoch': epoch + 1, 'model_state_dict': run_best_state_dict, 'optimizer_state_dict': optimizer.state_dict(), 'config': run_config, 'best_epoch': run_best_epoch, 'best_val_loss': run_best_val_loss, 'stale_epochs': run_stale_epochs, 'history': history}, run_output_dir / 'checkpoints' / 'best_model.pt')
                else:
                    run_stale_epochs += 1
                latest_checkpoint = {'epoch': epoch + 1, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'config': run_config, 'best_epoch': run_best_epoch, 'best_val_loss': run_best_val_loss, 'stale_epochs': run_stale_epochs, 'history': history}
                torch.save(latest_checkpoint, run_output_dir / 'checkpoints' / 'latest_checkpoint.pt')
                if checkpoint_every > 0 and (epoch + 1) % checkpoint_every == 0:
                    torch.save(latest_checkpoint, run_output_dir / 'checkpoints' / f'checkpoint_epoch_{epoch + 1}.pt')
                if patience > 0 and run_stale_epochs >= patience:
                    print(f'Early stopping at epoch {epoch + 1}. Best epoch: {run_best_epoch}, best val loss: {run_best_val_loss:.6f}')
                    break
            if run_best_state_dict is None:
                run_best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            torch.save({'epoch': len(history), 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'config': run_config, 'best_epoch': run_best_epoch, 'best_val_loss': run_best_val_loss, 'stale_epochs': run_stale_epochs, 'history': history}, run_output_dir / 'checkpoints' / 'last_model.pt')
            (run_output_dir / 'results' / 'history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')
            run_summary = {'dropout_prob': float(dropout_prob), 'best_epoch': run_best_epoch, 'best_val_loss': run_best_val_loss, 'epochs_ran': len(history), 'output_dir': run_output_dir.relative_to(REPO_ROOT).as_posix()}
            (run_output_dir / 'results' / 'summary.json').write_text(json.dumps(run_summary, indent=2), encoding='utf-8')
            sweep_histories[float(dropout_prob)] = history
            sweep_rows.append(run_summary)
            if best_run is None or run_best_val_loss < best_run['best_val_loss']:
                best_run = {'dropout_prob': float(dropout_prob), 'best_epoch': run_best_epoch, 'best_val_loss': run_best_val_loss, 'history': history, 'output_dir': run_output_dir, 'config': run_config}
        sweep_results_df = pd.DataFrame(sweep_rows).sort_values(['best_val_loss', 'epochs_ran']).reset_index(drop=True)
        best_dropout = float(best_run['dropout_prob'])
        config = best_run['config']
        output_dir = best_run['output_dir']
        history = best_run['history']
        history_path = output_dir / 'results' / 'history.json'
        best_epoch = int(best_run['best_epoch'])
        best_val_loss = float(best_run['best_val_loss'])
        best_model_path = output_dir / 'checkpoints' / 'best_model.pt'
        model = build_autoencoder(best_dropout)
        best_checkpoint = torch.load(best_model_path, map_location=device)
        model.load_state_dict(best_checkpoint['model_state_dict'])
        optimizer = torch.optim.Adam(model.parameters(), lr=float(config['training']['learning_rate']), weight_decay=float(config['training']['weight_decay']))
        best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        training_ran = True
        print(f'\nSelected dropout={best_dropout:.2f} from sweep')
    display(sweep_results_df)
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "epochs = int(config['training']['epochs'])\npatience = int(config['training'].get('early_stopping_patience', 0))\nmin_delta = float(config['training'].get('early_stopping_min_delta', 0.0))\ncheckpoint_every = int(config['training'].get('checkpoint_every', 5))\nbase_output_dir = repo_root / config['run']['output_dir']\nbase_output_dir.mkdir(parents=true, exist_ok=true)\nsweep_results_path = base_output_dir / 'dropout_sweep_results.csv'\nsweep_summary_path = base_output_dir / 'results' / 'dropout_sweep_summary.json'\ntraining_ran = false\nresume_from = ''\nsweep_histories = {}\nhistory = []\nbest_state_dict = none\nbest_epoch = 0\nbest_val_loss = float('inf')\nstale_epochs = 0\n\ndef resolve_run_output_dir(dropout_prob: float, stored_relative: str | none=none) -> path:\n    expected = base_output_dir / dropout_slug(dropout_prob)\n    if expected.exists() or not stored_relative:\n        return expected\n    candidate = repo_root / path(stored_relative)\n    if candidate.exists():\n        return candidate\n    return expected\nartifacts_ready = false\nif not retrain and sweep_results_path.exists() and sweep_summary_path.exists():\n    sweep_results_df = pd.read_csv(sweep_results_path)\n    sweep_summary = json.loads(sweep_summary_path.read_text(encoding='utf-8'))\n    best_dropout = float(sweep_summary.get('selected_dropout', sweep_results_df.sort_values('best_val_loss').iloc[0]['dropout_prob']))\n    normalized_rows = []\n    for row in sweep_results_df.to_dict(orient='records'):\n        dropout_prob = float(row['dropout_prob'])\n        run_dir = resolve_run_output_dir(dropout_prob, row.get('output_dir'))\n        row['dropout_prob'] = dropout_prob\n        row['output_dir'] = run_dir.relative_to(repo_root).as_posix()\n        normalized_rows.append(row)\n        history_candidate = run_dir / 'results' / 'history.json'\n        if history_candidate.exists():\n            sweep_histories[dropout_prob] = json.loads(history_candidate.read_text(encoding='utf-8'))\n    sweep_results_df = pd.dataframe(normalized_rows).sort_values(['best_val_loss', 'epochs_ran']).reset_index(drop=true)\n    output_dir = resolve_run_output_dir(best_dropout, sweep_summary.get('selected_output_dir'))\n    history_path = output_dir / 'results' / 'history.json'\n    best_model_path = output_dir / 'checkpoints' / 'best_model.pt'\n    artifacts_ready = history_path.exists() and best_model_path.exists()\n    if artifacts_ready:\n        history = json.loads(history_path.read_text(encoding='utf-8'))\n        config['model']['dropout_prob'] = best_dropout\n        config['run']['output_dir'] = output_dir.relative_to(repo_root).as_posix()\n        best_checkpoint = torch.load(best_model_path, map_location=device)\n        best_epoch = int(best_checkpoint.get('best_epoch', best_checkpoint.get('epoch', 0)))\n        best_val_loss = float(best_checkpoint.get('best_val_loss', float('nan')))\n        stale_epochs = int(best_checkpoint.get('stale_epochs', 0))\n        model = build_autoencoder(best_dropout)\n        model.load_state_dict(best_checkpoint['model_state_dict'])\n        optimizer = torch.optim.adam(model.parameters(), lr=float(config['training']['learning_rate']), weight_decay=float(config['training']['weight_decay']))\n        best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n        print(f'found existing sweep artifacts in {base_output_dir}. skipping sweep.')\n    else:\n        print('found sweep metadata, but the selected run artifacts are incomplete. rerunning the sweep.')\nif retrain or not artifacts_ready:\n    print({'epochs': epochs, 'anomaly_score': anomaly_score_name, 'topk_ratio': topk_ratio, 'dropout_sweep': dropout_sweep})\n    sweep_rows = []\n    sweep_histories = {}\n    best_run = none\n    for dropout_prob in dropout_sweep:\n        run_output_dir = base_output_dir / dropout_slug(dropout_prob)\n        run_output_dir.mkdir(parents=true, exist_ok=true)\n        run_config = json.loads(json.dumps(config))\n        run_config['model']['dropout_prob'] = float(dropout_"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Sweep Diagnostics

This cell visualizes the saved sweep histories, highlights the selected dropout setting, and saves the summary figure.

In [ ]:
try:
    history_df = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for dropout_prob, run_history in sweep_histories.items():
        run_history_df = pd.DataFrame(run_history)
        if not run_history_df.empty:
            axes[0].plot(run_history_df['epoch'], run_history_df['val_loss'], marker='o', label=f'dropout={dropout_prob:.2f}')
    axes[0].set_title('Validation Loss Across Dropout Sweep')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Val Loss')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    axes[1].bar(sweep_results_df['dropout_prob'].astype(str), sweep_results_df['best_val_loss'])
    axes[1].set_title('Best Validation Loss by Dropout')
    axes[1].set_xlabel('Dropout')
    axes[1].set_ylabel('Best Val Loss')
    plt.tight_layout()
    save_figure(fig, base_output_dir / 'plots' / 'dropout_sweep_summary.png')
    plt.show()
    print(f'Selected dropout for downstream evaluation: {best_dropout:.2f}')
    history_df.tail()
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "history_df = pd.dataframe(history)\nfig, axes = plt.subplots(1, 2, figsize=(14, 4))\nfor dropout_prob, run_history in sweep_histories.items():\n    run_history_df = pd.dataframe(run_history)\n    if not run_history_df.empty:\n        axes[0].plot(run_history_df['epoch'], run_history_df['val_loss'], marker='o', label=f'dropout={dropout_prob:.2f}')\naxes[0].set_title('validation loss across dropout sweep')\naxes[0].set_xlabel('epoch')\naxes[0].set_ylabel('val loss')\naxes[0].grid(true, alpha=0.3)\naxes[0].legend()\naxes[1].bar(sweep_results_df['dropout_prob'].astype(str), sweep_results_df['best_val_loss'])\naxes[1].set_title('best validation loss by dropout')\naxes[1].set_xlabel('dropout')\naxes[1].set_ylabel('best val loss')\nplt.tight_layout()\nsave_figure(fig, base_output_dir / 'plots' / 'dropout_sweep_summary.png')\nplt.show()\nprint(f'selected dropout for downstream evaluation: {best_dropout:.2f}')\nhistory_df.tail()"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Persist Sweep Outputs

This cell refreshes the sweep summary files so the artifact directory reflects the current selected run and paths.

In [ ]:
try:
    normalized_rows = []
    for row in sweep_results_df.to_dict(orient='records'):
        dropout_prob = float(row['dropout_prob'])
        run_dir = base_output_dir / dropout_slug(dropout_prob)
        row['dropout_prob'] = dropout_prob
        row['output_dir'] = run_dir.relative_to(REPO_ROOT).as_posix()
        normalized_rows.append(row)
    sweep_results_df = pd.DataFrame(normalized_rows).sort_values(['best_val_loss', 'epochs_ran']).reset_index(drop=True)
    sweep_summary = {'selected_dropout': best_dropout, 'selected_output_dir': output_dir.relative_to(REPO_ROOT).as_posix(), 'selection_metric': 'best_val_loss', 'runs': sweep_results_df.to_dict(orient='records')}
    sweep_results_df.to_csv(base_output_dir / 'dropout_sweep_results.csv', index=False)
    with (base_output_dir / 'results' / 'dropout_sweep_summary.json').open('w', encoding='utf-8') as handle:
        json.dump(sweep_summary, handle, indent=2)
    history_df.to_csv(output_dir / 'history.csv', index=False)
    if training_ran:
        print(f'Saved sweep outputs to {base_output_dir}')
    else:
        print(f'Confirmed existing sweep artifacts and refreshed metadata paths in {base_output_dir}')
    print(f'Selected run directory: {output_dir}')
    sweep_summary
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "normalized_rows = []\nfor row in sweep_results_df.to_dict(orient='records'):\n    dropout_prob = float(row['dropout_prob'])\n    run_dir = base_output_dir / dropout_slug(dropout_prob)\n    row['dropout_prob'] = dropout_prob\n    row['output_dir'] = run_dir.relative_to(repo_root).as_posix()\n    normalized_rows.append(row)\nsweep_results_df = pd.dataframe(normalized_rows).sort_values(['best_val_loss', 'epochs_ran']).reset_index(drop=true)\nsweep_summary = {'selected_dropout': best_dropout, 'selected_output_dir': output_dir.relative_to(repo_root).as_posix(), 'selection_metric': 'best_val_loss', 'runs': sweep_results_df.to_dict(orient='records')}\nsweep_results_df.to_csv(base_output_dir / 'dropout_sweep_results.csv', index=false)\nwith (base_output_dir / 'results' / 'dropout_sweep_summary.json').open('w', encoding='utf-8') as handle:\n    json.dump(sweep_summary, handle, indent=2)\nhistory_df.to_csv(output_dir / 'history.csv', index=false)\nif training_ran:\n    print(f'saved sweep outputs to {base_output_dir}')\nelse:\n    print(f'confirmed existing sweep artifacts and refreshed metadata paths in {base_output_dir}')\nprint(f'selected run directory: {output_dir}')\nsweep_summary"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Load Best Checkpoint And Score Test Split

This cell loads the best checkpoint from the selected dropout run and computes anomaly scores on the test split.

In [ ]:
try:
    best_model_path = output_dir / 'checkpoints' / 'best_model.pt'
    if not best_model_path.exists():
        raise FileNotFoundError(f'Best checkpoint not found at {best_model_path}')
    model = build_autoencoder(best_dropout)
    best_checkpoint = torch.load(best_model_path, map_location=device)
    model.load_state_dict(best_checkpoint['model_state_dict'])
    print(f"Loaded best_model.pt from epoch {best_checkpoint.get('best_epoch', 'unknown')} for dropout={best_dropout:.2f}")
    model.eval()

    def reconstruction_error(inputs: torch.Tensor, outputs: torch.Tensor, score_name: str=ANOMALY_SCORE_NAME) -> torch.Tensor:
        if score_name == 'mse_mean':
            return spatial_mean(squared_error_map(inputs, outputs))
        if score_name == 'topk_abs_mean':
            return topk_spatial_mean(absolute_error_map(inputs, outputs), topk_ratio=TOPK_RATIO)
        raise ValueError(f'Unsupported score_name: {score_name}')
    test_scores = []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            scores = reconstruction_error(inputs, outputs, score_name=ANOMALY_SCORE_NAME).cpu().numpy()
            labels = labels.cpu().numpy()
            for score, label in zip(scores, labels):
                test_scores.append({'score': float(score), 'is_anomaly': int(label)})
    score_df = pd.DataFrame(test_scores)
    print({'selected_dropout': best_dropout, 'evaluation_score': ANOMALY_SCORE_NAME, 'topk_ratio': TOPK_RATIO if ANOMALY_SCORE_NAME == 'topk_abs_mean' else None})
    score_df.head()
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = 'best_model_path = output_dir / \'checkpoints\' / \'best_model.pt\'\nif not best_model_path.exists():\n    raise filenotfounderror(f\'best checkpoint not found at {best_model_path}\')\nmodel = build_autoencoder(best_dropout)\nbest_checkpoint = torch.load(best_model_path, map_location=device)\nmodel.load_state_dict(best_checkpoint[\'model_state_dict\'])\nprint(f"loaded best_model.pt from epoch {best_checkpoint.get(\'best_epoch\', \'unknown\')} for dropout={best_dropout:.2f}")\nmodel.eval()\n\ndef reconstruction_error(inputs: torch.tensor, outputs: torch.tensor, score_name: str=anomaly_score_name) -> torch.tensor:\n    if score_name == \'mse_mean\':\n        return spatial_mean(squared_error_map(inputs, outputs))\n    if score_name == \'topk_abs_mean\':\n        return topk_spatial_mean(absolute_error_map(inputs, outputs), topk_ratio=topk_ratio)\n    raise valueerror(f\'unsupported score_name: {score_name}\')\ntest_scores = []\nwith torch.no_grad():\n    for inputs, labels in test_loader:\n        inputs = inputs.to(device)\n        outputs = model(inputs)\n        scores = reconstruction_error(inputs, outputs, score_name=anomaly_score_name).cpu().numpy()\n        labels = labels.cpu().numpy()\n        for score, label in zip(scores, labels):\n            test_scores.append({\'score\': float(score), \'is_anomaly\': int(label)})\nscore_df = pd.dataframe(test_scores)\nprint({\'selected_dropout\': best_dropout, \'evaluation_score\': anomaly_score_name, \'topk_ratio\': topk_ratio if anomaly_score_name == \'topk_abs_mean\' else none})\nscore_df.head()'
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Validation Threshold

This cell computes the deployment threshold from validation-normal scores.

In [ ]:
try:
    val_scores = []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            scores = reconstruction_error(inputs, outputs, score_name=ANOMALY_SCORE_NAME).cpu().numpy()
            val_scores.extend(scores.tolist())
    val_score_series = pd.Series(val_scores, name='val_score')
    threshold = float(val_score_series.quantile(0.95))
    print(f'Chosen threshold from validation normals (95th percentile, {ANOMALY_SCORE_NAME}): {threshold:.6f}')
    val_score_series.describe()
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "val_scores = []\nwith torch.no_grad():\n    for inputs, labels in val_loader:\n        inputs = inputs.to(device)\n        outputs = model(inputs)\n        scores = reconstruction_error(inputs, outputs, score_name=anomaly_score_name).cpu().numpy()\n        val_scores.extend(scores.tolist())\nval_score_series = pd.series(val_scores, name='val_score')\nthreshold = float(val_score_series.quantile(0.95))\nprint(f'chosen threshold from validation normals (95th percentile, {anomaly_score_name}): {threshold:.6f}')\nval_score_series.describe()"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Metrics

This cell applies the validation-derived threshold, computes evaluation metrics, and saves the score table and metric summary.

In [ ]:
try:
    score_df['predicted_anomaly'] = (score_df['score'] > threshold).astype(int)
    precision = precision_score(score_df['is_anomaly'], score_df['predicted_anomaly'], zero_division=0)
    recall = recall_score(score_df['is_anomaly'], score_df['predicted_anomaly'], zero_division=0)
    f1 = f1_score(score_df['is_anomaly'], score_df['predicted_anomaly'], zero_division=0)
    auroc = roc_auc_score(score_df['is_anomaly'], score_df['score'])
    auprc = average_precision_score(score_df['is_anomaly'], score_df['score'])
    cm = confusion_matrix(score_df['is_anomaly'], score_df['predicted_anomaly'])
    metrics_df = pd.DataFrame([{'metric': 'score_name', 'value': ANOMALY_SCORE_NAME}, {'metric': 'precision', 'value': precision}, {'metric': 'recall', 'value': recall}, {'metric': 'f1', 'value': f1}, {'metric': 'auroc', 'value': auroc}, {'metric': 'auprc', 'value': auprc}, {'metric': 'threshold', 'value': threshold}])
    display(metrics_df)
    cm_df = pd.DataFrame(cm, index=['true_normal', 'true_anomaly'], columns=['pred_normal', 'pred_anomaly'])
    display(cm_df)
    fig, ax = plt.subplots(figsize=(5, 4))
    heatmap = ax.imshow(cm_df.to_numpy(), cmap='Blues')
    ax.set_xticks(range(cm_df.shape[1]), cm_df.columns)
    ax.set_yticks(range(cm_df.shape[0]), cm_df.index)
    ax.set_title('Confusion Matrix At Validation Threshold')
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    for row_idx in range(cm_df.shape[0]):
        for col_idx in range(cm_df.shape[1]):
            value = int(cm_df.iat[row_idx, col_idx])
            ax.text(col_idx, row_idx, str(value), ha='center', va='center', color='black')
    fig.colorbar(heatmap, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    save_figure(fig, output_dir / 'plots' / 'confusion_matrix.png')
    plt.show()
    score_df.to_csv(output_dir / 'results' / 'test_scores.csv', index=False)
    metrics_df.to_csv(output_dir / 'results' / 'metrics.csv', index=False)
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "score_df['predicted_anomaly'] = (score_df['score'] > threshold).astype(int)\nprecision = precision_score(score_df['is_anomaly'], score_df['predicted_anomaly'], zero_division=0)\nrecall = recall_score(score_df['is_anomaly'], score_df['predicted_anomaly'], zero_division=0)\nf1 = f1_score(score_df['is_anomaly'], score_df['predicted_anomaly'], zero_division=0)\nauroc = roc_auc_score(score_df['is_anomaly'], score_df['score'])\nauprc = average_precision_score(score_df['is_anomaly'], score_df['score'])\ncm = confusion_matrix(score_df['is_anomaly'], score_df['predicted_anomaly'])\nmetrics_df = pd.dataframe([{'metric': 'score_name', 'value': anomaly_score_name}, {'metric': 'precision', 'value': precision}, {'metric': 'recall', 'value': recall}, {'metric': 'f1', 'value': f1}, {'metric': 'auroc', 'value': auroc}, {'metric': 'auprc', 'value': auprc}, {'metric': 'threshold', 'value': threshold}])\ndisplay(metrics_df)\ncm_df = pd.dataframe(cm, index=['true_normal', 'true_anomaly'], columns=['pred_normal', 'pred_anomaly'])\ndisplay(cm_df)\nfig, ax = plt.subplots(figsize=(5, 4))\nheatmap = ax.imshow(cm_df.to_numpy(), cmap='blues')\nax.set_xticks(range(cm_df.shape[1]), cm_df.columns)\nax.set_yticks(range(cm_df.shape[0]), cm_df.index)\nax.set_title('confusion matrix at validation threshold')\nax.set_xlabel('predicted label')\nax.set_ylabel('true label')\nfor row_idx in range(cm_df.shape[0]):\n    for col_idx in range(cm_df.shape[1]):\n        value = int(cm_df.iat[row_idx, col_idx])\n        ax.text(col_idx, row_idx, str(value), ha='center', va='center', color='black')\nfig.colorbar(heatmap, ax=ax, fraction=0.046, pad=0.04)\nplt.tight_layout()\nsave_figure(fig, output_dir / 'plots' / 'confusion_matrix.png')\nplt.show()\nscore_df.to_csv(output_dir / 'results' / 'test_scores.csv', index=false)\nmetrics_df.to_csv(output_dir / 'results' / 'metrics.csv', index=false)"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Threshold Sweep Plot

This cell compares precision, recall, and F1 across score thresholds, then saves both the table and the figure.

In [ ]:
try:
    precision_curve, recall_curve, pr_thresholds = precision_recall_curve(score_df['is_anomaly'], score_df['score'])
    threshold_sweep_df = pd.DataFrame({'threshold': pr_thresholds, 'precision': precision_curve[:-1], 'recall': recall_curve[:-1]})
    threshold_sweep_df['f1'] = 2 * threshold_sweep_df['precision'] * threshold_sweep_df['recall'] / (threshold_sweep_df['precision'] + threshold_sweep_df['recall'] + 1e-12)
    threshold_sweep_df['predicted_anomalies'] = [int((score_df['score'] > t).sum()) for t in threshold_sweep_df['threshold']]
    best_f1_row = threshold_sweep_df.loc[threshold_sweep_df['f1'].idxmax()]
    threshold_sweep_df.to_csv(output_dir / 'results' / 'threshold_sweep.csv', index=False)
    display(threshold_sweep_df.sort_values('f1', ascending=False).head(10))
    print(f"Best F1 threshold: {best_f1_row['threshold']:.6f} | precision={best_f1_row['precision']:.4f}, recall={best_f1_row['recall']:.4f}, f1={best_f1_row['f1']:.4f}")
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['precision'], label='precision')
    ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['recall'], label='recall')
    ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['f1'], label='f1')
    ax.axvline(threshold, color='red', linestyle='--', label=f'validation threshold = {threshold:.4f}')
    ax.axvline(best_f1_row['threshold'], color='green', linestyle=':', label=f"best f1 threshold = {best_f1_row['threshold']:.4f}")
    ax.set_xlabel('Anomaly-score threshold')
    ax.set_ylabel('Metric value')
    ax.set_title(f'Threshold Sweep on Test Split ({ANOMALY_SCORE_NAME})')
    ax.legend()
    save_figure(fig, output_dir / 'plots' / 'threshold_sweep.png')
    plt.show()
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = 'precision_curve, recall_curve, pr_thresholds = precision_recall_curve(score_df[\'is_anomaly\'], score_df[\'score\'])\nthreshold_sweep_df = pd.dataframe({\'threshold\': pr_thresholds, \'precision\': precision_curve[:-1], \'recall\': recall_curve[:-1]})\nthreshold_sweep_df[\'f1\'] = 2 * threshold_sweep_df[\'precision\'] * threshold_sweep_df[\'recall\'] / (threshold_sweep_df[\'precision\'] + threshold_sweep_df[\'recall\'] + 1e-12)\nthreshold_sweep_df[\'predicted_anomalies\'] = [int((score_df[\'score\'] > t).sum()) for t in threshold_sweep_df[\'threshold\']]\nbest_f1_row = threshold_sweep_df.loc[threshold_sweep_df[\'f1\'].idxmax()]\nthreshold_sweep_df.to_csv(output_dir / \'results\' / \'threshold_sweep.csv\', index=false)\ndisplay(threshold_sweep_df.sort_values(\'f1\', ascending=false).head(10))\nprint(f"best f1 threshold: {best_f1_row[\'threshold\']:.6f} | precision={best_f1_row[\'precision\']:.4f}, recall={best_f1_row[\'recall\']:.4f}, f1={best_f1_row[\'f1\']:.4f}")\nfig, ax = plt.subplots(figsize=(8, 4))\nax.plot(threshold_sweep_df[\'threshold\'], threshold_sweep_df[\'precision\'], label=\'precision\')\nax.plot(threshold_sweep_df[\'threshold\'], threshold_sweep_df[\'recall\'], label=\'recall\')\nax.plot(threshold_sweep_df[\'threshold\'], threshold_sweep_df[\'f1\'], label=\'f1\')\nax.axvline(threshold, color=\'red\', linestyle=\'--\', label=f\'validation threshold = {threshold:.4f}\')\nax.axvline(best_f1_row[\'threshold\'], color=\'green\', linestyle=\':\', label=f"best f1 threshold = {best_f1_row[\'threshold\']:.4f}")\nax.set_xlabel(\'anomaly-score threshold\')\nax.set_ylabel(\'metric value\')\nax.set_title(f\'threshold sweep on test split ({anomaly_score_name})\')\nax.legend()\nsave_figure(fig, output_dir / \'plots\' / \'threshold_sweep.png\')\nplt.show()'
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Score Distribution Plot

This cell visualizes the test-score distribution for normal and anomalous wafers and saves the histogram figure.

In [ ]:
try:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(score_df[score_df['is_anomaly'] == 0]['score'], bins=30, alpha=0.7, label='normal')
    ax.hist(score_df[score_df['is_anomaly'] == 1]['score'], bins=30, alpha=0.7, label='anomaly')
    ax.axvline(threshold, color='red', linestyle='--', label=f'threshold={threshold:.4f}')
    ax.set_title(f'Anomaly Score on Test Split ({ANOMALY_SCORE_NAME})')
    ax.set_xlabel(f'Per-sample score: {ANOMALY_SCORE_NAME}')
    ax.set_ylabel('Count')
    ax.legend()
    plt.tight_layout()
    save_figure(fig, output_dir / 'plots' / 'score_distribution.png')
    plt.show()
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "fig, ax = plt.subplots(figsize=(8, 4))\nax.hist(score_df[score_df['is_anomaly'] == 0]['score'], bins=30, alpha=0.7, label='normal')\nax.hist(score_df[score_df['is_anomaly'] == 1]['score'], bins=30, alpha=0.7, label='anomaly')\nax.axvline(threshold, color='red', linestyle='--', label=f'threshold={threshold:.4f}')\nax.set_title(f'anomaly score on test split ({anomaly_score_name})')\nax.set_xlabel(f'per-sample score: {anomaly_score_name}')\nax.set_ylabel('count')\nax.legend()\nplt.tight_layout()\nsave_figure(fig, output_dir / 'plots' / 'score_distribution.png')\nplt.show()"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Reconstruction Examples

This cell shows a small set of input and reconstruction pairs and saves the figure.

In [ ]:
try:
    normal_test_idx = score_df[score_df['is_anomaly'] == 0].index[:4].tolist()
    anomaly_test_idx = score_df[score_df['is_anomaly'] == 1].index[:4].tolist()
    selected_indices = normal_test_idx + anomaly_test_idx
    fig, axes = plt.subplots(len(selected_indices), 2, figsize=(6, 2.3 * len(selected_indices)))
    with torch.no_grad():
        for row_idx, sample_idx in enumerate(selected_indices):
            input_tensor, label = test_dataset[sample_idx]
            output_tensor = model(input_tensor.unsqueeze(0).to(device)).squeeze(0).cpu()
            title_prefix = 'anomaly' if int(label) == 1 else 'normal'
            score = score_df.iloc[sample_idx]['score']
            axes[row_idx, 0].imshow(input_tensor.squeeze(0), cmap='viridis')
            axes[row_idx, 0].set_title(f'Input: {title_prefix}')
            axes[row_idx, 0].axis('off')
            axes[row_idx, 1].imshow(output_tensor.squeeze(0), cmap='viridis')
            axes[row_idx, 1].set_title(f'Recon, score={score:.4f}')
            axes[row_idx, 1].axis('off')
    plt.tight_layout()
    save_figure(fig, output_dir / 'plots' / 'reconstruction_examples.png')
    plt.show()
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "normal_test_idx = score_df[score_df['is_anomaly'] == 0].index[:4].tolist()\nanomaly_test_idx = score_df[score_df['is_anomaly'] == 1].index[:4].tolist()\nselected_indices = normal_test_idx + anomaly_test_idx\nfig, axes = plt.subplots(len(selected_indices), 2, figsize=(6, 2.3 * len(selected_indices)))\nwith torch.no_grad():\n    for row_idx, sample_idx in enumerate(selected_indices):\n        input_tensor, label = test_dataset[sample_idx]\n        output_tensor = model(input_tensor.unsqueeze(0).to(device)).squeeze(0).cpu()\n        title_prefix = 'anomaly' if int(label) == 1 else 'normal'\n        score = score_df.iloc[sample_idx]['score']\n        axes[row_idx, 0].imshow(input_tensor.squeeze(0), cmap='viridis')\n        axes[row_idx, 0].set_title(f'input: {title_prefix}')\n        axes[row_idx, 0].axis('off')\n        axes[row_idx, 1].imshow(output_tensor.squeeze(0), cmap='viridis')\n        axes[row_idx, 1].set_title(f'recon, score={score:.4f}')\n        axes[row_idx, 1].axis('off')\nplt.tight_layout()\nsave_figure(fig, output_dir / 'plots' / 'reconstruction_examples.png')\nplt.show()"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Failure Tables

This cell builds the error-analysis table and saves the detailed failure-analysis CSVs for later reference.

In [ ]:
try:
    analysis_df = test_dataset.metadata.reset_index(drop=True).copy()
    analysis_df = pd.concat([analysis_df, score_df[['score', 'predicted_anomaly']].reset_index(drop=True)], axis=1)
    analysis_df['error_type'] = 'tn'
    analysis_df.loc[(analysis_df['is_anomaly'] == 0) & (analysis_df['predicted_anomaly'] == 1), 'error_type'] = 'fp'
    analysis_df.loc[(analysis_df['is_anomaly'] == 1) & (analysis_df['predicted_anomaly'] == 0), 'error_type'] = 'fn'
    analysis_df.loc[(analysis_df['is_anomaly'] == 1) & (analysis_df['predicted_anomaly'] == 1), 'error_type'] = 'tp'
    analysis_df['correct'] = analysis_df['is_anomaly'] == analysis_df['predicted_anomaly']
    error_summary_df = analysis_df.groupby('error_type').agg(count=('error_type', 'size'), mean_score=('score', 'mean')).reindex(['tp', 'fn', 'fp', 'tn'])
    defect_recall_df = analysis_df[analysis_df['is_anomaly'] == 1].groupby('defect_type').agg(count=('defect_type', 'size'), detected=('predicted_anomaly', 'sum'), mean_score=('score', 'mean')).sort_values(['detected', 'count'], ascending=[False, False])
    defect_recall_df['recall'] = defect_recall_df['detected'] / defect_recall_df['count']
    fp_defect_df = analysis_df[analysis_df['error_type'] == 'fp'].groupby('defect_type').agg(count=('defect_type', 'size'), mean_score=('score', 'mean')).sort_values(['count', 'mean_score'], ascending=[False, False])
    display(error_summary_df)
    display(defect_recall_df)
    display(fp_defect_df)
    analysis_df.head()
    analysis_df.to_csv(output_dir / 'results' / 'failure_analysis.csv', index=False)
    error_summary_df.to_csv(output_dir / 'results' / 'failure_error_summary.csv')
    defect_recall_df.to_csv(output_dir / 'results' / 'failure_defect_recall.csv')
    fp_defect_df.to_csv(output_dir / 'results' / 'failure_false_positive_breakdown.csv')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "analysis_df = test_dataset.metadata.reset_index(drop=true).copy()\nanalysis_df = pd.concat([analysis_df, score_df[['score', 'predicted_anomaly']].reset_index(drop=true)], axis=1)\nanalysis_df['error_type'] = 'tn'\nanalysis_df.loc[(analysis_df['is_anomaly'] == 0) & (analysis_df['predicted_anomaly'] == 1), 'error_type'] = 'fp'\nanalysis_df.loc[(analysis_df['is_anomaly'] == 1) & (analysis_df['predicted_anomaly'] == 0), 'error_type'] = 'fn'\nanalysis_df.loc[(analysis_df['is_anomaly'] == 1) & (analysis_df['predicted_anomaly'] == 1), 'error_type'] = 'tp'\nanalysis_df['correct'] = analysis_df['is_anomaly'] == analysis_df['predicted_anomaly']\nerror_summary_df = analysis_df.groupby('error_type').agg(count=('error_type', 'size'), mean_score=('score', 'mean')).reindex(['tp', 'fn', 'fp', 'tn'])\ndefect_recall_df = analysis_df[analysis_df['is_anomaly'] == 1].groupby('defect_type').agg(count=('defect_type', 'size'), detected=('predicted_anomaly', 'sum'), mean_score=('score', 'mean')).sort_values(['detected', 'count'], ascending=[false, false])\ndefect_recall_df['recall'] = defect_recall_df['detected'] / defect_recall_df['count']\nfp_defect_df = analysis_df[analysis_df['error_type'] == 'fp'].groupby('defect_type').agg(count=('defect_type', 'size'), mean_score=('score', 'mean')).sort_values(['count', 'mean_score'], ascending=[false, false])\ndisplay(error_summary_df)\ndisplay(defect_recall_df)\ndisplay(fp_defect_df)\nanalysis_df.head()\nanalysis_df.to_csv(output_dir / 'results' / 'failure_analysis.csv', index=false)\nerror_summary_df.to_csv(output_dir / 'results' / 'failure_error_summary.csv')\ndefect_recall_df.to_csv(output_dir / 'results' / 'failure_defect_recall.csv')\nfp_defect_df.to_csv(output_dir / 'results' / 'failure_false_positive_breakdown.csv')"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Failure Examples

This cell visualizes representative false positives, false negatives, true positives, and true negatives and saves each figure.

In [ ]:
def show_error_examples(error_type: str, n_examples: int=6, score_order: str='desc') -> pd.DataFrame:
    subset = analysis_df[analysis_df['error_type'] == error_type].copy()
    if subset.empty:
        print(f'No samples found for error_type={error_type!r}.')
        return subset
    ascending = score_order == 'asc'
    subset = subset.sort_values('score', ascending=ascending).head(n_examples)
    fig, axes = plt.subplots(len(subset), 3, figsize=(9, 2.8 * len(subset)))
    if len(subset) == 1:
        axes = np.expand_dims(axes, axis=0)
    with torch.no_grad():
        for row_idx, (sample_idx, row) in enumerate(subset.iterrows()):
            input_tensor, label = test_dataset[sample_idx]
            output_tensor = model(input_tensor.unsqueeze(0).to(device)).squeeze(0).cpu()
            error_map = absolute_error_map(input_tensor.unsqueeze(0), output_tensor.unsqueeze(0)).squeeze(0).squeeze(0).cpu()
            axes[row_idx, 0].imshow(input_tensor.squeeze(0), cmap='viridis')
            axes[row_idx, 0].set_title(f"Input\nlabel={int(label)} | defect={row.get('defect_type', 'unknown')}")
            axes[row_idx, 0].axis('off')
            axes[row_idx, 1].imshow(output_tensor.squeeze(0), cmap='viridis')
            axes[row_idx, 1].set_title(f"Recon\nscore={row['score']:.4f} | pred={row['predicted_anomaly']}")
            axes[row_idx, 1].axis('off')
            axes[row_idx, 2].imshow(error_map, cmap='magma')
            axes[row_idx, 2].set_title(f'Abs error map\n{error_type.upper()} sample #{sample_idx}')
            axes[row_idx, 2].axis('off')
    plt.tight_layout()
    save_figure(fig, output_dir / 'plots' / f'failure_examples_{error_type}.png')
    plt.show()
    return subset[['defect_type', 'score', 'predicted_anomaly', 'error_type']]
display(show_error_examples('fp', n_examples=6, score_order='desc'))
display(show_error_examples('fn', n_examples=6, score_order='asc'))
display(show_error_examples('tp', n_examples=4, score_order='desc'))
display(show_error_examples('tn', n_examples=4, score_order='asc'))

### Score Ablation Run

This cell runs the score-ablation helper only when its outputs are missing or rerun is explicitly requested.

In [ ]:
try:
    import subprocess
    score_ablation_config = config if 'config' in globals() else load_toml(CONFIG_PATH)
    score_ablation_output_root = REPO_ROOT / score_ablation_config['run']['output_dir']
    score_ablation_best_model_path = score_ablation_output_root / 'checkpoints' / 'best_model.pt'
    if not score_ablation_best_model_path.exists():
        raise FileNotFoundError(f'Best autoencoder checkpoint not found: {score_ablation_best_model_path}')
    score_ablation_output_dir = score_ablation_output_root / 'results' / 'score_ablation'
    score_ablation_output_dir.mkdir(parents=True, exist_ok=True)
    score_ablation_csv_path = score_ablation_output_dir / 'score_summary.csv'
    score_ablation_json_path = score_ablation_output_dir / 'score_summary.json'
    score_ablation_cmd = [sys.executable, 'scripts/evaluate_autoencoder_scores.py', '--checkpoint', str(score_ablation_best_model_path.relative_to(REPO_ROOT)), '--config', str(CONFIG_PATH.relative_to(REPO_ROOT)), '--output-dir', str(score_ablation_output_dir.relative_to(REPO_ROOT))]
    if RERUN_SCORE_ABLATION:
        print('Running:')
        print(' '.join(score_ablation_cmd))
        subprocess.run(score_ablation_cmd, cwd=REPO_ROOT, check=True)
    elif score_ablation_csv_path.exists() and score_ablation_json_path.exists():
        print(f'Found existing score ablation outputs in {score_ablation_output_dir}. Skipping rerun.')
    else:
        print('[WARNING] The rerun/training flags are False and the saved artifacts for this section are missing. Skipping this section.')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "import subprocess\nscore_ablation_config = config if 'config' in globals() else load_toml(config_path)\nscore_ablation_output_root = repo_root / score_ablation_config['run']['output_dir']\nscore_ablation_best_model_path = score_ablation_output_root / 'checkpoints' / 'best_model.pt'\nif not score_ablation_best_model_path.exists():\n    raise filenotfounderror(f'best autoencoder checkpoint not found: {score_ablation_best_model_path}')\nscore_ablation_output_dir = score_ablation_output_root / 'results' / 'score_ablation'\nscore_ablation_output_dir.mkdir(parents=true, exist_ok=true)\nscore_ablation_csv_path = score_ablation_output_dir / 'score_summary.csv'\nscore_ablation_json_path = score_ablation_output_dir / 'score_summary.json'\nscore_ablation_cmd = [sys.executable, 'scripts/evaluate_autoencoder_scores.py', '--checkpoint', str(score_ablation_best_model_path.relative_to(repo_root)), '--config', str(config_path.relative_to(repo_root)), '--output-dir', str(score_ablation_output_dir.relative_to(repo_root))]\nif rerun_score_ablation:\n    print('running:')\n    print(' '.join(score_ablation_cmd))\n    subprocess.run(score_ablation_cmd, cwd=repo_root, check=true)\nelif score_ablation_csv_path.exists() and score_ablation_json_path.exists():\n    print(f'found existing score ablation outputs in {score_ablation_output_dir}. skipping rerun.')\nelse:\n    print('[warning] the rerun/training flags are false and the saved artifacts for this section are missing. skipping this section.')"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Score Ablation Results

This cell loads the saved score-ablation outputs so they can be inspected without rerunning the script.

In [ ]:
try:
    score_ablation_df = pd.read_csv(score_ablation_output_dir / 'score_summary.csv')
    score_ablation_summary = json.loads((score_ablation_output_dir / 'score_summary.json').read_text(encoding='utf-8'))
    display(score_ablation_df)
    score_ablation_summary
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "score_ablation_df = pd.read_csv(score_ablation_output_dir / 'score_summary.csv')\nscore_ablation_summary = json.loads((score_ablation_output_dir / 'score_summary.json').read_text(encoding='utf-8'))\ndisplay(score_ablation_df)\nscore_ablation_summary"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Score Ablation Plot

This cell visualizes the score-ablation comparison and saves the summary plot.

In [ ]:
top_scores = score_ablation_df.sort_values('val_threshold_f1', ascending=False).reset_index(drop=True)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].bar(top_scores['score_name'], top_scores['val_threshold_f1'])
axes[0].set_title('Validation-Threshold F1')
axes[0].tick_params(axis='x', rotation=35)
axes[1].bar(top_scores['score_name'], top_scores['auroc'])
axes[1].set_title('AUROC')
axes[1].tick_params(axis='x', rotation=35)
axes[2].bar(top_scores['score_name'], top_scores['auprc'])
axes[2].set_title('AUPRC')
axes[2].tick_params(axis='x', rotation=35)
plt.tight_layout()
save_figure(fig, score_ablation_output_root / 'plots' / 'score_ablation_summary.png')
plt.show()
top_scores[['score_name', 'val_threshold_f1', 'auroc', 'auprc', 'best_sweep_f1']]

---

## Holdout Evaluation: Expanded 70k Normal / 3.5k Defect Test Set

This section evaluates the saved checkpoint on the secondary holdout split.

The holdout keeps the same 40k training normals and 5k validation normals as the main
benchmark, but replaces the test set with a much larger pool:
**70,000 normal + 3,500 defect** wafers.

This is a robustness check - the checkpoint and threshold policy do not change.

Note: `reconstruction_error` inside this notebook only supports `topk_abs_mean` and `mse_mean`.
To evaluate `max_abs` on the holdout, use `scripts/evaluate_autoencoder_scores.py` directly.

In [ ]:
RUN_HOLDOUT_EVALUATION = False
HOLDOUT_METADATA_PATH = REPO_ROOT / 'data/processed/x64/wm811k/metadata_50k_5pct_holdout70k_3p5k.csv'
HOLDOUT_SCORE_NAME = 'topk_abs_mean'
HOLDOUT_THRESHOLD_QUANTILE = 0.95
HOLDOUT_OUTPUT_DIR = output_dir / 'holdout70k_3p5k'
print(f'Holdout metadata: {HOLDOUT_METADATA_PATH}')
print(f'Exists:           {HOLDOUT_METADATA_PATH.exists()}')
print(f'Score:            {HOLDOUT_SCORE_NAME}')
print(f'Output dir:       {HOLDOUT_OUTPUT_DIR}')

In [ ]:
try:
    if not RUN_HOLDOUT_EVALUATION:
        print('Holdout evaluation skipped.')
    else:
        HOLDOUT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        print('Loading holdout datasets...')
        holdout_val_ds = WaferMapDataset(HOLDOUT_METADATA_PATH, split='val', image_size=image_size)
        holdout_test_ds = WaferMapDataset(HOLDOUT_METADATA_PATH, split='test', image_size=image_size)
        holdout_val_loader = DataLoader(holdout_val_ds, batch_size=64, shuffle=False, num_workers=0)
        holdout_test_loader = DataLoader(holdout_test_ds, batch_size=64, shuffle=False, num_workers=0)
        print(f'  Val:  {len(holdout_val_ds):,} wafers')
        print(f'  Test: {len(holdout_test_ds):,} wafers')
        model.eval()
        holdout_val_scores, holdout_val_labels = ([], [])
        with torch.no_grad():
            for imgs, labels in holdout_val_loader:
                imgs = imgs.to(device)
                recon = model(imgs)
                scores = reconstruction_error(imgs, recon, score_name=HOLDOUT_SCORE_NAME).cpu().numpy()
                holdout_val_scores.extend(scores.tolist())
                holdout_val_labels.extend(labels.tolist())
        holdout_val_scores = np.array(holdout_val_scores)
        holdout_val_labels = np.array(holdout_val_labels)
        holdout_threshold = float(np.quantile(holdout_val_scores[holdout_val_labels == 0], HOLDOUT_THRESHOLD_QUANTILE))
        print(f'\nHoldout threshold ({HOLDOUT_THRESHOLD_QUANTILE:.0%} of val normals): {holdout_threshold:.6f}')
        holdout_test_scores, holdout_test_labels = ([], [])
        with torch.no_grad():
            for imgs, labels in holdout_test_loader:
                imgs = imgs.to(device)
                recon = model(imgs)
                scores = reconstruction_error(imgs, recon, score_name=HOLDOUT_SCORE_NAME).cpu().numpy()
                holdout_test_scores.extend(scores.tolist())
                holdout_test_labels.extend(labels.tolist())
        holdout_test_scores = np.array(holdout_test_scores)
        holdout_test_labels = np.array(holdout_test_labels)
        holdout_test_preds = (holdout_test_scores >= holdout_threshold).astype(int)
        h_precision = precision_score(holdout_test_labels, holdout_test_preds, zero_division=0)
        h_recall = recall_score(holdout_test_labels, holdout_test_preds, zero_division=0)
        h_f1 = f1_score(holdout_test_labels, holdout_test_preds, zero_division=0)
        h_auroc = roc_auc_score(holdout_test_labels, holdout_test_scores)
        h_auprc = average_precision_score(holdout_test_labels, holdout_test_scores)
        print(f'\n-- Holdout Evaluation Results -------------------------------')
        print(f'  Score:               {HOLDOUT_SCORE_NAME}')
        print(f'  Threshold:           {holdout_threshold:.6f}')
        print(f'  Precision:           {h_precision:.6f}')
        print(f'  Recall:              {h_recall:.6f}')
        print(f'  F1:                  {h_f1:.6f}')
        print(f'  AUROC:               {h_auroc:.6f}')
        print(f'  AUPRC:               {h_auprc:.6f}')
        print(f'  Test normals:        {int((holdout_test_labels == 0).sum()):,}')
        print(f'  Test anomalies:      {int((holdout_test_labels == 1).sum()):,}')
        print(f'  Predicted anomalies: {int(holdout_test_preds.sum()):,}')
        holdout_meta = holdout_test_ds.metadata.reset_index(drop=True).copy()
        holdout_meta['score'] = holdout_test_scores
        holdout_meta['predicted'] = holdout_test_preds
        defect_col = next((c for c in ['defect_type', 'failureType'] if c in holdout_meta.columns), None)
        if defect_col:
            holdout_defect_df = holdout_meta[holdout_meta['is_anomaly'] == 1].groupby(defect_col).apply(lambda g: pd.Series({'count': len(g), 'detected': int(g['predicted'].sum()), 'recall': g['predicted'].mean()})).reset_index().sort_values('recall')
            print(f'\n-- Per-Defect Recall ----------------------------------------')
            display(holdout_defect_df)
        else:
            holdout_defect_df = pd.DataFrame()
            print('\nNo defect type column found in metadata; per-defect recall skipped.')
        h_prec_curve, h_rec_curve, h_thresholds = precision_recall_curve(holdout_test_labels, holdout_test_scores)
        holdout_sweep_df = pd.DataFrame({'threshold': h_thresholds, 'precision': h_prec_curve[:-1], 'recall': h_rec_curve[:-1]})
        holdout_sweep_df['f1'] = 2 * holdout_sweep_df['precision'] * holdout_sweep_df['recall'] / (holdout_sweep_df['precision'] + holdout_sweep_df['recall'] + 1e-12)
        best_h_row = holdout_sweep_df.loc[holdout_sweep_df['f1'].idxmax()]
        print(f"\nBest sweep F1: {best_h_row['f1']:.6f} at threshold {best_h_row['threshold']:.6f}")
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        axes[0].hist(holdout_test_scores[holdout_test_labels == 0], bins=50, alpha=0.7, label='normal')
        axes[0].hist(holdout_test_scores[holdout_test_labels == 1], bins=50, alpha=0.7, label='anomaly')
        axes[0].axvline(holdout_threshold, color='red', linestyle='--', label=f'threshold={holdout_threshold:.4f}')
        axes[0].set_title(f'Holdout Score Distribution ({HOLDOUT_SCORE_NAME})')
        axes[0].set_xlabel('Score')
        axes[0].set_ylabel('Count')
        axes[0].legend()
        axes[1].plot(holdout_sweep_df['threshold'], holdout_sweep_df['precision'], label='precision')
        axes[1].plot(holdout_sweep_df['threshold'], holdout_sweep_df['recall'], label='recall')
        axes[1].plot(holdout_sweep_df['threshold'], holdout_sweep_df['f1'], label='f1')
        axes[1].axvline(holdout_threshold, color='red', linestyle='--', label=f'val threshold={holdout_threshold:.4f}')
        axes[1].axvline(best_h_row['threshold'], color='green', linestyle=':', label=f"best f1={best_h_row['threshold']:.4f}")
        axes[1].set_title('Holdout Threshold Sweep')
        axes[1].set_xlabel('Threshold')
        axes[1].legend()
        plt.tight_layout()
        save_figure(fig, HOLDOUT_OUTPUT_DIR / 'holdout_distribution_sweep.png')
        plt.show()
        pd.DataFrame({'score': holdout_val_scores, 'is_anomaly': holdout_val_labels}).to_csv(HOLDOUT_OUTPUT_DIR / 'val_scores.csv', index=False)
        holdout_meta[['score', 'is_anomaly', 'predicted']].to_csv(HOLDOUT_OUTPUT_DIR / 'test_scores.csv', index=False)
        holdout_sweep_df.to_csv(HOLDOUT_OUTPUT_DIR / 'threshold_sweep.csv', index=False)
        if not holdout_defect_df.empty:
            holdout_defect_df.to_csv(HOLDOUT_OUTPUT_DIR / 'defect_recall.csv', index=False)
        pd.DataFrame([{'score_name': HOLDOUT_SCORE_NAME, 'threshold': holdout_threshold, 'threshold_quantile': HOLDOUT_THRESHOLD_QUANTILE, 'precision': h_precision, 'recall': h_recall, 'f1': h_f1, 'auroc': h_auroc, 'auprc': h_auprc, 'best_sweep_f1': float(best_h_row['f1']), 'best_sweep_threshold': float(best_h_row['threshold']), 'test_normal_count': int((holdout_test_labels == 0).sum()), 'test_anomaly_count': int((holdout_test_labels == 1).sum()), 'predicted_anomalies': int(holdout_test_preds.sum())}]).to_csv(HOLDOUT_OUTPUT_DIR / 'summary.csv', index=False)
        print(f'\nResults saved to {HOLDOUT_OUTPUT_DIR}')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "if not run_holdout_evaluation:\n    print('holdout evaluation skipped.')\nelse:\n    holdout_output_dir.mkdir(parents=true, exist_ok=true)\n    print('loading holdout datasets...')\n    holdout_val_ds = wafermapdataset(holdout_metadata_path, split='val', image_size=image_size)\n    holdout_test_ds = wafermapdataset(holdout_metadata_path, split='test', image_size=image_size)\n    holdout_val_loader = dataloader(holdout_val_ds, batch_size=64, shuffle=false, num_workers=0)\n    holdout_test_loader = dataloader(holdout_test_ds, batch_size=64, shuffle=false, num_workers=0)\n    print(f'  val:  {len(holdout_val_ds):,} wafers')\n    print(f'  test: {len(holdout_test_ds):,} wafers')\n    model.eval()\n    holdout_val_scores, holdout_val_labels = ([], [])\n    with torch.no_grad():\n        for imgs, labels in holdout_val_loader:\n            imgs = imgs.to(device)\n            recon = model(imgs)\n            scores = reconstruction_error(imgs, recon, score_name=holdout_score_name).cpu().numpy()\n            holdout_val_scores.extend(scores.tolist())\n            holdout_val_labels.extend(labels.tolist())\n    holdout_val_scores = np.array(holdout_val_scores)\n    holdout_val_labels = np.array(holdout_val_labels)\n    holdout_threshold = float(np.quantile(holdout_val_scores[holdout_val_labels == 0], holdout_threshold_quantile))\n    print(f'\\nholdout threshold ({holdout_threshold_quantile:.0%} of val normals): {holdout_threshold:.6f}')\n    holdout_test_scores, holdout_test_labels = ([], [])\n    with torch.no_grad():\n        for imgs, labels in holdout_test_loader:\n            imgs = imgs.to(device)\n            recon = model(imgs)\n            scores = reconstruction_error(imgs, recon, score_name=holdout_score_name).cpu().numpy()\n            holdout_test_scores.extend(scores.tolist())\n            holdout_test_labels.extend(labels.tolist())\n    holdout_test_scores = np.array(holdout_test_scores)\n    holdout_test_labels = np.array(holdout_test_labels)\n    holdout_test_preds = (holdout_test_scores >= holdout_threshold).astype(int)\n    h_precision = precision_score(holdout_test_labels, holdout_test_preds, zero_division=0)\n    h_recall = recall_score(holdout_test_labels, holdout_test_preds, zero_division=0)\n    h_f1 = f1_score(holdout_test_labels, holdout_test_preds, zero_division=0)\n    h_auroc = roc_auc_score(holdout_test_labels, holdout_test_scores)\n    h_auprc = average_precision_score(holdout_test_labels, holdout_test_scores)\n    print(f'\\n-- holdout evaluation results -------------------------------')\n    print(f'  score:               {holdout_score_name}')\n    print(f'  threshold:           {holdout_threshold:.6f}')\n    print(f'  precision:           {h_precision:.6f}')\n    print(f'  recall:              {h_recall:.6f}')\n    print(f'  f1:                  {h_f1:.6f}')\n    print(f'  auroc:               {h_auroc:.6f}')\n    print(f'  auprc:               {h_auprc:.6f}')\n    print(f'  test normals:        {int((holdout_test_labels == 0).sum()):,}')\n    print(f'  test anomalies:      {int((holdout_test_labels == 1).sum()):,}')\n    print(f'  predicted anomalies: {int(holdout_test_preds.sum()):,}')\n    holdout_meta = holdout_test_ds.metadata.reset_index(drop=true).copy()\n    holdout_meta['score'] = holdout_test_scores\n    holdout_meta['predicted'] = holdout_test_preds\n    defect_col = next((c for c in ['defect_type', 'failuretype'] if c in holdout_meta.columns), none)\n    if defect_col:\n        holdout_defect_df = holdout_meta[holdout_meta['is_anomaly'] == 1].groupby(defect_col).apply(lambda g: pd.series({'count': len(g), 'detected': int(g['predicted'].sum()), 'recall': g['predicted'].mean()})).reset_index().sort_values('recall')\n        print(f'\\n-- per-defect recall ----------------------------------------')\n        display(holdout_defect_df)\n    else:\n        holdout_defect_df = pd.dataframe()\n        print('\\nno defect type column found in metadata; per-defect recall skipped.')\n    h_prec_curve, h_rec_curve, h_thresholds = precision_recall_curve(holdout_test"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise
